<a href="https://colab.research.google.com/github/samueluribe27/Calidad_y_mineria_de_datos/blob/main/03_despliegue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gradio
from google.colab import files
import pickle, pandas as pd
from sklearn.preprocessing import LabelEncoder

uploaded = files.upload()

Saving mejor_modelo.pkl to mejor_modelo (1).pkl
Saving datos_limpios.csv to datos_limpios.csv


In [3]:
with open('mejor_modelo.pkl', 'rb') as f:
    modelo = pickle.load(f)

df = pd.read_csv('datos_limpios.csv')

le_tema = LabelEncoder().fit(df['tema'])
le_alcalde = LabelEncoder().fit(df['alcalde'])

temas = sorted(df['tema'].unique().tolist())
alcaldes = sorted(df['alcalde'].unique().tolist())

In [4]:
import gradio as gr

def predecir(vigencia, articulado, tema, alcalde):
    tema_enc = le_tema.transform([tema])[0]
    alcalde_enc = le_alcalde.transform([alcalde])[0]
    entrada = [[vigencia, articulado, tema_enc, alcalde_enc]]
    pred = modelo.predict(entrada)[0]
    prob = modelo.predict_proba(entrada)[0]
    resultado = "Vigente" if pred == 1 else "Derogado"
    confianza = f"{max(prob)*100:.1f}%"
    return resultado, confianza

interfaz = gr.Interface(
    fn=predecir,
    inputs=[
        gr.Slider(2008, 2026, step=1, label="Año de vigencia"),
        gr.Slider(1, 50, step=1, label="Número de artículos"),
        gr.Dropdown(choices=temas, label="Tema"),
        gr.Dropdown(choices=alcaldes, label="Alcalde")
    ],
    outputs=[
        gr.Text(label="Predicción"),
        gr.Text(label="Confianza del modelo")
    ],
    title="Predictor de Estado de Acuerdos Distritales - Cartagena",
    description="Predice si un acuerdo distrital quedará Vigente o Derogado"
)

interfaz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc80c7a4ac8a687a71.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
